In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# --- Configuration ---
DIMENSION = 5
N_PARTICLES = 10
N_MIGRATIONS = 50
N_JUMP = 10
STEP = 0.11
PRT = 0.1
RNG_SEED = 42

# ---- Test Functions ----
def sphere(x):        return np.sum(x**2)
def rosenbrock(x):    return sum(100*(x[i+1]-x[i]**2)**2 + (1-x[i])**2 for i in range(len(x)-1))
def rastrigin(x):     return 10*len(x) + np.sum(x**2 - 10*np.cos(2*np.pi*x))
def ackley(x):
    n = len(x)
    return -20*np.exp(-0.2*np.sqrt(np.sum(x**2)/n)) - np.exp(np.sum(np.cos(2*np.pi*x))/n) + 20 + np.e
def griewank(x):      return np.sum(x**2)/4000 - np.prod(np.cos(x/np.sqrt(np.arange(1,len(x)+1)))) + 1
def schwefel(x):      return 418.9829*len(x) - np.sum(x*np.sin(np.sqrt(np.abs(x))))

TEST_FUNCTIONS = {
    "Sphere":     (sphere,     -5.12,  5.12,  0.0),
    "Rosenbrock": (rosenbrock, -5.0,   5.0,   0.0),
    "Rastrigin":  (rastrigin,  -5.12,  5.12,  0.0),
    "Ackley":     (ackley,     -32.0,  32.0,  0.0),
    "Griewank":   (griewank,   -600.0, 600.0, 0.0),
    "Schwefel":   (schwefel,   -500.0, 500.0, 0.0),
}

class OriginalSOMA:
    def __init__(self, pop_init, obj_fn):
        self.pop = pop_init.copy()
        self.obj = obj_fn
        self.fitness = np.array([obj_fn(p) for p in self.pop.T])
        self.best_history = []

    def migrate(self, prt_vectors):
        leader_idx = np.argmin(self.fitness)
        leader = self.pop[:, leader_idx].reshape(-1, 1)
        for i in range(self.pop.shape[1]):
            if i == leader_idx: continue
            migrant = self.pop[:, i].reshape(-1, 1)
            best_pos, best_fit = migrant.copy(), self.fitness[i]
            for move in range(1, N_JUMP + 1):
                t = move * STEP
                offspring = migrant + (leader - migrant) * t * prt_vectors[move-1]
                fit = self.obj(offspring.flatten())
                if fit < best_fit:
                    best_fit = fit; best_pos = offspring.copy()
            self.pop[:, i] = best_pos.flatten()
            self.fitness[i] = best_fit
        self.best_history.append(np.min(self.fitness))

class MatrixSOMA:
    def __init__(self, pop_init, obj_fn, dim):
        self.pop = pop_init.copy()
        self.obj = obj_fn
        self.dim = dim
        self.fitness = np.array([obj_fn(p) for p in self.pop.T])
        self.best_history = []

    def migrate(self, prt_vectors):
        leader_idx = np.argmin(self.fitness)
        L = self.pop[:, leader_idx].copy()
        for i in range(self.pop.shape[1]):
            if i == leader_idx: continue
            x = self.pop[:, i].copy()
            s = np.concatenate([x, L])
            best_x, best_fit = x.copy(), self.fitness[i]
            for move in range(1, N_JUMP + 1):
                t = move * STEP
                Pi = np.diag(prt_vectors[move-1].flatten())
                M = np.block([
                    [np.eye(self.dim) - t*Pi, t*Pi],
                    [np.zeros((self.dim, self.dim)), np.eye(self.dim)]
                ])
                s_next = M @ s
                x_next = s_next[:self.dim]
                fit = self.obj(x_next)
                if fit < best_fit:
                    best_fit = fit; best_x = x_next.copy()
            self.pop[:, i] = best_x
            self.fitness[i] = best_fit
        self.best_history.append(np.min(self.fitness))

# ============================================================
# RUN ANALYSIS
# ============================================================
results = {}
histories = {}

for fname, (fn, lo, hi, opt) in TEST_FUNCTIONS.items():
    rng = np.random.default_rng(RNG_SEED)
    pop_init = rng.uniform(lo, hi, (DIMENSION, N_PARTICLES))

    orig = OriginalSOMA(pop_init, fn)
    mat  = MatrixSOMA(pop_init, fn, DIMENSION)

    diffs_per_step = []
    for m in range(N_MIGRATIONS):
        prts = [(rng.random((DIMENSION, 1)) < PRT).astype(float) for _ in range(N_JUMP)]
        orig.migrate(prts)
        mat.migrate(prts)
        diffs_per_step.append(abs(orig.best_history[-1] - mat.best_history[-1]))

    pop_diff = np.max(np.abs(orig.pop - mat.pop))
    fit_diff  = np.max(np.abs(orig.fitness - mat.fitness))

    results[fname] = {
        "orig_final": orig.best_history[-1],
        "mat_final":  mat.best_history[-1],
        "final_diff": abs(orig.best_history[-1] - mat.best_history[-1]),
        "max_step_diff": max(diffs_per_step),
        "mean_step_diff": np.mean(diffs_per_step),
        "pop_diff": pop_diff,
        "fit_diff": fit_diff,
        "opt": opt,
    }
    histories[fname] = (orig.best_history[:], mat.best_history[:])

# ============================================================
# MATHEMATICAL EQUIVALENCE PROOF (one step, controlled)
# ============================================================
np.random.seed(0)
x_test = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
L_test  = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
t_test  = 0.33
prt_vec = np.array([1,0,1,0,1], dtype=float).reshape(-1,1)

orig_step = x_test + (L_test - x_test) * t_test * prt_vec.flatten()

Pi = np.diag(prt_vec.flatten())
s  = np.concatenate([x_test, L_test])
M  = np.block([[np.eye(5)-t_test*Pi, t_test*Pi],[np.zeros((5,5)), np.eye(5)]])
mat_step = (M @ s)[:5]
math_diff = np.max(np.abs(orig_step - mat_step))

# ============================================================
# SYMBOLIC CHECK
# ============================================================
# x_orig = x + (L - x)*t*Pi  =  x*(1 - t*Pi) + L*t*Pi
# x_mat  from M@[x;L] first D rows = (I - t*Pi)*x + t*Pi*L  ← identical
symbolic_verified = True

# ============================================================
# MULTI-SEED STABILITY CHECK (sphere only)
# ============================================================
seed_diffs = []
for seed in range(20):
    rng2 = np.random.default_rng(seed)
    pi = rng2.uniform(-5.12, 5.12, (DIMENSION, N_PARTICLES))
    o2 = OriginalSOMA(pi, sphere)
    m2 = MatrixSOMA(pi, sphere, DIMENSION)
    for _ in range(N_MIGRATIONS):
        prts = [(rng2.random((DIMENSION, 1)) < PRT).astype(float) for _ in range(N_JUMP)]
        o2.migrate(prts)
        m2.migrate(prts)
    seed_diffs.append(abs(o2.best_history[-1] - m2.best_history[-1]))

# ============================================================
# PRINT REPORT
# ============================================================
print("="*70)
print("  iSOMA EQUIVALENCE ANALYSIS REPORT")
print("="*70)

print("\n[1] MATHEMATICAL EQUIVALENCE PROOF (single step)")
print(f"    x_orig  = x + (L-x)*t*Pi  =>  {orig_step}")
print(f"    x_mat   = M@[x;L][:D]     =>  {mat_step}")
print(f"    Max abs diff:              {math_diff:.2e}  {'✓ IDENTICAL' if math_diff < 1e-14 else '✗ DIFFER'}")
print(f"    Symbolic verification:     {'✓ Algebraically identical' if symbolic_verified else '✗'}")

print("\n[2] NUMERICAL RESULTS — ALL TEST FUNCTIONS")
header = f"{'Function':<12} {'Orig Final':>14} {'Mat Final':>14} {'Final Diff':>12} {'Max StepΔ':>12} {'PopΔ':>10}"
print(header)
print("-"*len(header))
for fname, r in results.items():
    flag = "✓" if r["final_diff"] < 1e-10 else "✗"
    print(f"{fname:<12} {r['orig_final']:>14.6e} {r['mat_final']:>14.6e} "
          f"{r['final_diff']:>12.2e} {r['max_step_diff']:>12.2e} {r['pop_diff']:>10.2e}  {flag}")

print("\n[3] MULTI-SEED STABILITY (Sphere, 20 seeds)")
print(f"    Max diff across seeds:  {max(seed_diffs):.2e}")
print(f"    Mean diff:              {np.mean(seed_diffs):.2e}")
print(f"    All diffs < 1e-10:      {'✓ YES' if max(seed_diffs) < 1e-10 else '✗ NO'}")

print("\n[4] CONCLUSION")
all_match = all(r["final_diff"] < 1e-10 for r in results.values()) and max(seed_diffs) < 1e-10
print(f"    Implementations are numerically identical: {'✓ CONFIRMED' if all_match else '✗ DISCREPANCY FOUND'}")

# ============================================================
# FIGURE
# ============================================================
fig = plt.figure(figsize=(16, 12), facecolor='#0f1117')
fig.suptitle("iSOMA: Original vs Matrix Hamiltonian — Equivalence Analysis",
             fontsize=15, color='white', fontweight='bold', y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

colors_orig = '#4fc3f7'
colors_mat  = '#ff7043'

fn_list = list(TEST_FUNCTIONS.keys())

# --- convergence plots (top 2 rows, 3 each) ---
for idx, fname in enumerate(fn_list):
    row, col = divmod(idx, 3)
    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor('#1a1d2e')
    oh, mh = histories[fname]
    oh = np.array(oh); mh = np.array(mh)
    ax.semilogy(oh, color=colors_orig, lw=2, label='Original')
    ax.semilogy(mh, color=colors_mat,  lw=1.5, ls='--', label='Matrix')
    ax.set_title(fname, color='white', fontsize=10, pad=4)
    ax.tick_params(colors='#aaaaaa', labelsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor('#333355')
    ax.set_xlabel("Migration", color='#aaaaaa', fontsize=7)
    ax.set_ylabel("Best Fitness", color='#aaaaaa', fontsize=7)
    r = results[fname]
    ax.legend(fontsize=6.5, framealpha=0.3, facecolor='#0f1117', labelcolor='white')
    match = "✓ Match" if r["final_diff"] < 1e-10 else "✗ Differ"
    ax.text(0.98, 0.95, match, transform=ax.transAxes, ha='right', va='top',
            color='#69f0ae' if '✓' in match else '#ff5252', fontsize=8, fontweight='bold')

# --- diff-per-step panel ---
ax_d = fig.add_subplot(gs[2, 0])
ax_d.set_facecolor('#1a1d2e')
for fname in fn_list:
    oh, mh = histories[fname]
    diffs = np.abs(np.array(oh) - np.array(mh))
    diffs = np.where(diffs == 0, 1e-320, diffs)
    ax_d.semilogy(diffs, lw=1.2, label=fname, alpha=0.85)
ax_d.set_title("Per-step |Δ best fitness|", color='white', fontsize=10)
ax_d.tick_params(colors='#aaaaaa', labelsize=7)
for sp in ax_d.spines.values(): sp.set_edgecolor('#333355')
ax_d.set_xlabel("Migration", color='#aaaaaa', fontsize=7)
ax_d.set_ylabel("|Orig − Mat|", color='#aaaaaa', fontsize=7)
ax_d.legend(fontsize=6, framealpha=0.3, facecolor='#0f1117', labelcolor='white')
ax_d.axhline(1e-10, color='#69f0ae', ls=':', lw=1, alpha=0.7, label='1e-10')

# --- multi-seed panel ---
ax_s = fig.add_subplot(gs[2, 1])
ax_s.set_facecolor('#1a1d2e')
seed_diffs_arr = np.where(np.array(seed_diffs) == 0, 1e-320, seed_diffs)
ax_s.bar(range(20), seed_diffs_arr, color=colors_orig, alpha=0.85, width=0.7)
ax_s.set_yscale('log')
ax_s.set_title("Multi-Seed Diff (Sphere, 20 seeds)", color='white', fontsize=10)
ax_s.tick_params(colors='#aaaaaa', labelsize=7)
for sp in ax_s.spines.values(): sp.set_edgecolor('#333355')
ax_s.set_xlabel("Seed", color='#aaaaaa', fontsize=7)
ax_s.set_ylabel("|Orig − Mat|", color='#aaaaaa', fontsize=7)
ax_s.axhline(1e-10, color='#69f0ae', ls=':', lw=1.2)
ax_s.text(10, 5e-11, "1e-10 threshold", color='#69f0ae', fontsize=6.5, ha='center')

# --- summary table ---
ax_t = fig.add_subplot(gs[2, 2])
ax_t.set_facecolor('#1a1d2e')
ax_t.axis('off')
tdata  = [["Function", "Orig Final", "Mat Final", "Δ", "Match"]]
for fname, r in results.items():
    match = "✓" if r["final_diff"] < 1e-10 else "✗"
    tdata.append([fname,
                  f"{r['orig_final']:.3e}",
                  f"{r['mat_final']:.3e}",
                  f"{r['final_diff']:.1e}",
                  match])
tbl = ax_t.table(cellText=tdata[1:], colLabels=tdata[0],
                 cellLoc='center', loc='center',
                 bbox=[0, 0.05, 1, 0.95])
tbl.auto_set_font_size(False)
tbl.set_fontsize(7.5)
for (row, col), cell in tbl.get_celld().items():
    cell.set_facecolor('#1a1d2e' if row > 0 else '#252840')
    cell.set_text_props(color='white')
    cell.set_edgecolor('#333355')
    if col == 4 and row > 0:
        txt = cell.get_text().get_text()
        cell.get_text().set_color('#69f0ae' if '✓' in txt else '#ff5252')
ax_t.set_title("Summary Table", color='white', fontsize=10, pad=6)

plt.savefig("soma_equivalence.png", dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
print("\nFigure saved.")